# Data Science for Business - GAM on Ames Housing

*Session 6 · ISLP 7.5 Smoothing Splines · 7.7 GAMs · Lab 7.8.3 (pygam)*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
# Install packages that are not pre-installed on Colab
# !pip install "pygam>=0.10"

In [ ]:
import warnings

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

import statsmodels.formula.api as smf

from pygam import LinearGAM, s

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

# pygam's summary() always warns that its p-values are too small; we don't use them, so we silence only this warning
warnings.filterwarnings('ignore', message='KNOWN BUG: p-values', category=UserWarning)

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_06/ameshousing.csv')

In [ ]:
data.head()

## Prepare data

Let us focus on the numerical variables only.

In [ ]:
data = data.select_dtypes(include='number')

In [ ]:
data.head()

Finally, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets.

In [ ]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.head()

## Fit a linear regression as the baseline

To assess the performance of the GAM, we first fit a simple baseline model: a linear regression with two predictors, *YearBuilt* and *LotArea*. (ISLP's labs specify the same model with `ModelSpec` and `sm.OLS`; only the syntax differs.)

In [ ]:
mod_01 = smf.ols(formula='SalePrice ~ YearBuilt + LotArea', data=pd.concat([X_train, y_train], axis=1))
mod_01 = mod_01.fit()
print(mod_01.summary())

Make predictions on the test set and calculate the RMSE.

In [ ]:
y_pred = mod_01.predict(X_test)
root_mean_squared_error(y_test, y_pred)

## Fit a GAM

A generalized additive model (GAM, ISLP 7.7) keeps the additive form $y = \beta_0 + f_1(x_1) + f_2(x_2) + \varepsilon$ but lets each $f_j$ be a smooth non-linear function.

Here, both $f_j$ are smoothing splines (ISLP 7.5), specified with `s()`. Each term has a roughness penalty λ: a small λ gives a wiggly fit (risk of overfitting), a large λ pushes the function towards a straight line (ISLP 7.5.1). Strictly speaking, pygam approximates a smoothing spline: instead of a knot at every observation, it uses a penalised basis of 20 cubic spline functions (`n_splines=20`, `spline_order=3` by default).

pygam is not very user friendly: the terms refer to features by their column *index*, not their name. We therefore look up the index of each column by its name.

In [ ]:
X_train.columns

In [ ]:
gam_mod = LinearGAM(s(X_train.columns.get_loc('LotArea')) + s(X_train.columns.get_loc('YearBuilt')))
gam_mod.fit(X_train, y_train)

Let's have a look at the model. Due to the non-linearity, there are no coefficients to interpret, but the summary contains useful statistics:

- **Lambda**: the smoothing penalty of each term.
- **EDoF**: the effective degrees of freedom of each term, the book's $df_\lambda$ (ISLP 7.5.2, eq. 7.13). The higher, the more flexible (wiggly) the function.
- **Pseudo R-Squared** and **GCV** (see below).

Ignore the p-values: pygam itself warns that they are too small when λ is estimated.

In [ ]:
gam_mod.summary()

**GCV** (generalized cross-validation) is a fast approximation of leave-one-out cross-validation, computed from the training fit alone. It is not covered in ISLP, but it is closely related to the LOOCV shortcut for smoothing splines in ISLP 7.5.2. Its square root is on the RMSE scale, but it is an *estimate* computed from the training data, **not** a test error.

In [ ]:
np.sqrt(gam_mod.statistics_['GCV'])

To get a real test error, we predict on the test set, exactly as for the baseline.

In [ ]:
y_pred_gam = gam_mod.predict(X_test)
root_mean_squared_error(y_test, y_pred_gam)

The GAM explains more of the variance on the training data (pseudo R² vs. the R² of the linear regression) and also has a lower test RMSE than the linear regression baseline: allowing non-linear effects pays off here.

Let's try to interpret the patterns the model has learned. For non-linear models, we typically do that visually. Below, we create partial dependence plots (incl. 95% confidence intervals) for all terms of the model (cf. ISLP Figs. 7.11/7.12). Each plot shows one feature's contribution to the predicted price, holding the other feature fixed. The dashed lines are the 95% bands; they are wide where data are sparse.

In [ ]:
for i, term in enumerate(gam_mod.terms):
    if term.isintercept:
        continue

    x_grid = gam_mod.generate_X_grid(term=i)
    pdep, confi = gam_mod.partial_dependence(term=i, X=x_grid, width=0.95)

    feature_name = X_train.columns[term.feature]

    plt.figure()
    plt.plot(x_grid[:, term.feature], pdep)
    plt.plot(x_grid[:, term.feature], confi, c='r', ls='--')
    plt.xlabel(feature_name)
    plt.ylabel('Partial effect on SalePrice')
    plt.title(f'{feature_name} (lambda = {term.lam[0]:.3g})')
    plt.show()

## Tune the smoothness penalty

For fitting the model above, we have used the default value for the smoothing parameter λ (`lam`).

In [ ]:
print(gam_mod.lam)

Let's perform a grid search over multiple λ values to see if we can improve our model. We search for the model with the lowest GCV score. We try 9 values for each smoothing parameter, evenly spaced on a log scale from $10^{-2}$ to $10^{6}$ (the range used in Lab 7.8.3), resulting in a total of 9 × 9 = 81 points in our grid.

(ISLP chooses λ by LOOCV (7.5.2) or by fixing the effective degrees of freedom (Lab 7.8.3); pygam's GCV is a fast stand-in for LOOCV.)

In [ ]:
# perform grid search for optimal lambda values (gam_mod is updated with the best model)
lam = np.logspace(-2, 6, 9)
lams = [lam, lam]
gam_mod.gridsearch(X_train, y_train, lam=lams, objective='GCV', progress=False)

In [ ]:
# extract best lambda values
gam_mod.lam

In [ ]:
gam_mod.summary()

Calculate the square root of the GCV score of the tuned model.

In [ ]:
np.sqrt(gam_mod.statistics_['GCV'])

And now the real test: the RMSE on the test set.

In [ ]:
y_pred_gam = gam_mod.predict(X_test)
root_mean_squared_error(y_test, y_pred_gam)

Interesting: the tuning lowered the GCV score slightly, but the test RMSE got slightly *worse* (it is still better than the linear baseline). The grid search chose a much smaller λ for LotArea, i.e., a wigglier curve. Few houses have very large lots, so in that region the curve is driven by a handful of training points, and test houses with large lots get unreliable predictions. GCV is computed from the training data only; a lower GCV score is no guarantee of a lower test error. That's why we always confirm on held-out data.

Finally, let's visualize the effect of the tuning on the learned relationships.

In [ ]:
for i, term in enumerate(gam_mod.terms):
    if term.isintercept:
        continue

    x_grid = gam_mod.generate_X_grid(term=i)
    pdep, confi = gam_mod.partial_dependence(term=i, X=x_grid, width=0.95)

    feature_name = X_train.columns[term.feature]

    plt.figure()
    plt.plot(x_grid[:, term.feature], pdep)
    plt.plot(x_grid[:, term.feature], confi, c='r', ls='--')
    plt.xlabel(feature_name)
    plt.ylabel('Partial effect on SalePrice')
    plt.title(f'{feature_name} (lambda = {term.lam[0]:.3g})')
    plt.show()

## Your turn!

**Your turn!**

1. Choose two other predictors, fit a `LinearGAM`, plot the partial dependences and compare the test RMSE with the models above. Which relationship is clearly non-linear?
2. The LotArea curve is very sensitive to a few huge lots. Try to make it more robust, e.g., by fixing a larger λ for this term (`s(..., lam=1000)`) or by using `np.log(LotArea)` as predictor. Does the test RMSE improve?

In [ ]:
# YOUR CODE HERE